In [56]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List, operator

In [57]:
load_dotenv()

True

In [58]:
model = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=1
)

In [59]:
essay = '''
# The Dual-Edged Sword: Exploring the Pros and Cons of Artificial Intelligence

Artificial Intelligence (AI) has progressed from a staple of science fiction to the defining technology of the 21st century. By simulating human cognitive functions like learning, reasoning, and problem-solving, AI is fundamentally restructuring how we work, communicate, and live. Yet, as its integration into society deepens, it becomes clear that AI is a dual-edged sword. While it offers unprecedented opportunities for innovation and economic growth, it simultaneously introduces profound ethical, social, and economic risks. Navigating this technological shift requires a balanced understanding of its immense benefits against its significant drawbacks.

---

## The Pros: Driving Efficiency and Innovation

The primary allure of artificial intelligence lies in its ability to process data, automate complex workflows, and solve problems at a scale and speed that human beings simply cannot match.

### 1. Unprecedented Efficiency and Automation

At its core, AI excels at handling repetitive, time-consuming, and data-heavy tasks. In industrial settings, smart robotics optimize manufacturing pipelines, while in office environments, AI tools automate administrative duties like data entry, scheduling, and basic customer support. By absorbing these mundane tasks, AI reduces human error and frees human workers to focus on higher-order, strategic, and creative endeavors. Furthermore, unlike human labor, AI systems operate 24/7 without fatigue, drastically increasing overall productivity.

### 2. Revolutions in Healthcare and Research

Perhaps the most human-centric benefit of AI is its impact on medicine. AI algorithms can analyze massive datasets—ranging from medical imaging to genetic sequences—to detect anomalies with a high degree of accuracy. For example, machine learning models are routinely used to identify early-stage cancers in X-rays and MRIs long before they might be visible to a human radiologist. Additionally, AI has shortened the drug discovery pipeline from decades to mere months by simulating molecular interactions, paving the way for rapid breakthroughs in treating rare diseases.

### 3. Enhanced Data Analysis and Decision Making

In an era dominated by "Big Data," businesses and governments generate more information than human analysts can realistically parse. AI can look at millions of data points simultaneously to identify patterns, predict market trends, detect fraudulent financial activity, and optimize supply chains. This data-driven precision shifts decision-making from intuitive guesswork to predictive certainty.

---

## The Cons: Disruption, Bias, and Ethical Fragility

Despite its transformative potential, the rapid deployment of AI has outpaced our social safeguards, creating structural vulnerabilities that threaten societal stability.

### 1. Economic Disruption and Job Displacement

The automation that drives corporate efficiency is simultaneously a major catalyst for economic anxiety. As AI systems become more capable, they threaten not just blue-collar manufacturing roles, but white-collar professions in law, finance, coding, and journalism. While technology historically creates new jobs to replace old ones, the speed and scope of the AI transition threaten to displace workers faster than they can reskill, potentially widening the gap of economic inequality.

### 2. Algorithmic Bias and Discrimination

A common misconception is that because AI is mathematical, it is inherently objective. In reality, AI models learn from historical data generated by humans. If the training data contains historical biases, the AI will internalize, amplify, and codify those biases. This has already led to documented discrimination in high-stakes areas like predictive policing, automated hiring tools, and loan approvals, where AI systems systematically disadvantaged marginalized groups.

### 3. Misinformation, Security, and Intellectual Erosion

The rise of generative AI has made it incredibly easy to manufacture convincing fake text, audio, and video ("deepfakes"). This poses an existential threat to digital trust, making scams more sophisticated and enabling targeted disinformation campaigns that can destabilize democratic elections. On an individual level, over-reliance on AI for writing, thinking, and analysis risks eroding critical human skills, creating a dependency on "black box" algorithms whose internal logic we do not fully understand.

---

## Conclusion: Crafting a Supervised Future

Artificial Intelligence is neither inherently good nor fundamentally evil; it is a mirror of the data we feed it and the intentions of those who build it. The advantages of AI—such as life-saving medical breakthroughs, optimized resource management, and the elimination of dangerous labor—are too valuable to abandon. However, the risks of unchecked job displacement, systemic bias, and the erosion of digital truth are too severe to ignore.

The ultimate impact of AI will not be determined by the technology itself, but by governance. To ensure AI serves human progress rather than undermining it, society must establish robust regulatory frameworks, invest heavily in workforce retraining, and prioritize ethical design. The future of AI should not be a replacement for humanity, but a carefully managed collaboration that elevates human potential.

'''

In [60]:
class feedbackSchema(BaseModel):
    feedback: str = Field(description="Feedback of the essay")
    score: float = Field(description="Score of the essay out of 10", ge=0,le=10)

In [71]:
structured_model = model.with_structured_output(schema=feedbackSchema)

In [72]:
prompt = f'''
You are an expert in rating and judging blogs. Your task is to give a 1 oneliner feedback about the essay - {essay} and also provide the score out of 10.
'''

In [73]:
result = structured_model.invoke(prompt)

In [74]:
result.feedback

"Well-structured, balanced, and insightful overview of AI's benefits and pitfalls, though it could benefit from more concrete examples and citations."

In [84]:
class ContentFeed(TypedDict):
    essay: str
    clarity_feedback: str
    credibilty_feedback: str
    relevence_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[float], operator.add]
    
    avg_scores: float

In [91]:
def clarity(state: ContentFeed) -> ContentFeed:
    essay = state['essay']
    
    prompt = f"""
    You are an expert writing coach evaluating structural logic and communication flow.
    Analyze the essay provided below. 
    
    CRITICAL INSTRUCTION: You must strictly map your findings to the provided output formatting tools. 
    - Compress your structural analysis, logic gap observations, and sentence precision critiques into a single cohesive paragraph inside the 'feedback' field.
    - Assign an overall numeric score between 0 and 10 inside the 'score' field.

    Do not write any markdown sections or introductions outside the required fields.

    Essay to evaluate:
    {essay}
    """
    output = structured_model.invoke(prompt)
    return {'clarity_feedback' : output.feedback, "individual_scores": [output.score]}

In [92]:
def relevance(state: ContentFeed) -> ContentFeed:
    essay = state['essay']
    
    prompt = f"""
You are an academic evaluator analyzing alignment and scope. Your sole task is to evaluate the provided essay for Relevance based on the original prompt or topic description.

Please evaluate the text based on the following:
1. **Prompt Alignment:** Does every paragraph actively work to answer the core question, or does the writer drift into interesting but unnecessary tangents?
2. **Argumentative Weight:** Are the supporting points used actually advancing the main thesis, or do they feel like "filler" meant to hit a word count?
3. **Scope Control:** Did the author try to tackle an entire universe of ideas, or is the focus sharp and manageable for the length of the essay?

Provide a score from 1 to 10 for Relevance. To help the writer, list any paragraph or sentence that feels like a detour and should be cut entirely to make the paper tighter.


{essay}
    
    """
    output = structured_model.invoke(prompt)
    
    return {'relevence_feedback' : output.feedback, "individual_scores": [output.score]}

In [93]:
def credibilty(state: ContentFeed) -> ContentFeed:
    essay = state['essay']
    
    prompt = f"""
You are a rigorous fact-checker and peer reviewer. Your sole task is to evaluate the provided essay for Credibility, Intellectual Honesty, and Evidence.

Please evaluate the text based on the following:
1. **Evidentiary Support:** Are the major claims backed by verifiable facts, data, academic references, or ironclad logical deductions? Point out any "naked claims" (assertions made without proof).
2. **Nuance vs. Bias:** Does the writer acknowledge the complexity of the topic and address potential counterarguments, or is the tone overly one-sided, emotional, or dogmatic?
3. **Source Authority:** Do the sources cited actually support the specific claims the author is making, or is the author stretching the evidence to fit their narrative?

Provide a score from 1 to 10 for Credibility. List the three weakest or most vulnerable claims in the essay and explain what kind of evidence is missing to make them believable.

{essay}
    
    """
    output = structured_model.invoke(prompt)
    
    return {'credibilty_feedback' : output.feedback, "individual_scores": [output.score]}

In [94]:
def summary(state: ContentFeed) -> ContentFeed:
    prompt = f"Based on the 3 feedbacks create a summarized feedback 1. {state['clarity_feedback']}, 2. {state['credibilty_feedback']}, 3. {state['relevence_feedback']}"
    output = model.invoke(prompt)
    avg_score = sum(state['individual_scores'])/3
    
    return {'overall_feedback': output.content, 'avg_scores': avg_score}

In [95]:
graph = StateGraph(ContentFeed)

graph.add_node("clarity", clarity)
graph.add_node("credibilty", credibilty)
graph.add_node("relevance", relevance)
graph.add_node("summary", summary)

graph.add_edge(START, "clarity")
graph.add_edge(START, "credibilty")
graph.add_edge(START, "relevance")
graph.add_edge("clarity", "summary")
graph.add_edge("credibilty", "summary")
graph.add_edge("relevance", "summary")
graph.add_edge("summary", END)

workflow = graph.compile()




In [96]:
initial_state = {
    'essay' : essay
    }

final_state = workflow.invoke(initial_state)

In [97]:
final_state

{'essay': '\n# The Dual-Edged Sword: Exploring the Pros and Cons of Artificial Intelligence\n\nArtificial Intelligence (AI) has progressed from a staple of science fiction to the defining technology of the 21st century. By simulating human cognitive functions like learning, reasoning, and problem-solving, AI is fundamentally restructuring how we work, communicate, and live. Yet, as its integration into society deepens, it becomes clear that AI is a dual-edged sword. While it offers unprecedented opportunities for innovation and economic growth, it simultaneously introduces profound ethical, social, and economic risks. Navigating this technological shift requires a balanced understanding of its immense benefits against its significant drawbacks.\n\n---\n\n## The Pros: Driving Efficiency and Innovation\n\nThe primary allure of artificial intelligence lies in its ability to process data, automate complex workflows, and solve problems at a scale and speed that human beings simply cannot ma